# Interface with `LibAenet`

This notebook demonstrates how to use `LibAenetInterface` for direct energy and force evaluation with aenet potentials from Python. It also shows how the same potential can be used through `ANNPotential`, `TorchANNPotential`, and ASE's calculator interface.

**Requirements:**
- `libaenet` must be available from the aenet Fortran build
- the example ASCII potentials in `./pt-TiO2/`
- ASE for the optimization and MD sections

For a lightweight and reproducible example, the workflow below uses only a relatively small periodic TiO$_2$ cell.


In [1]:
import numpy as np
import aenet.io.structure

cell_path = "./TiO2-cell.xsf"
potential_paths = {
    "Ti": "./pt-TiO2/potential.Ti.nn.ascii",
    "O": "./pt-TiO2/potential.O.nn.ascii",
}
torch_model_path = "./pt-TiO2/Ti-O.pt"
outdir = "./example-08-outputs"

cell_structure = aenet.io.structure.read(str(cell_path))

## 1. Direct use of `LibAenetInterface`

`LibAenetInterface` provides direct access to `libaenet` from Python. This is useful when you want aenet's Fortran backend without going through the command-line tools.


In [2]:
from aenet.mlip import LibAenetInterface

interface = LibAenetInterface(potential_paths, potential_format="ascii")
energy_lib, forces_lib = interface.predict(cell_structure, forces=True)

print(f"LibAenetInterface energy: {energy_lib:.12f} eV")
print(f"Maximum |force|: {np.abs(forces_lib).max():.6f} eV/Ang")

LibAenetInterface energy: -19517.694574626403 eV
Maximum |force|: 2.803398 eV/Ang


## 2. Cross-check against the other interfaces

The same TiO$_2$ potential can also be evaluated through `ANNPotential`, `TorchANNPotential`, and ASE's `AenetCalculator`. The energies and forces should agree to within numerical precision.


In [3]:
from aenet.mlip import AenetCalculator, ANNPotential, LibAenetInterface
from aenet.torch_training.trainer import TorchANNPotential
import ase.io

ann_potential = ANNPotential.from_files(
    potential_paths,
    potential_format="ascii",
)
ann_results = ann_potential.predict([str(cell_path)], eval_forces=True)

torch_potential = TorchANNPotential.from_file(str(torch_model_path))
torch_results = torch_potential.predict([cell_structure], eval_forces=True)

calculator = AenetCalculator(potential_paths, potential_format="ascii")
cell_atoms = ase.io.read(cell_path)
atoms_for_eval = cell_atoms.copy()
atoms_for_eval.calc = calculator
energy_ase = atoms_for_eval.get_potential_energy()
forces_ase = atoms_for_eval.get_forces()

energy_ann = ann_results.total_energy[0]
forces_ann = ann_results.forces[0]
energy_torch = torch_results.total_energy[0]
forces_torch = torch_results.forces[0]

summary = {
    "LibAenetInterface": (energy_lib, forces_lib),
    "ANNPotential": (energy_ann, forces_ann),
    "TorchANNPotential": (energy_torch, forces_torch),
    "AenetCalculator": (energy_ase, forces_ase),
}

for name, (energy, forces) in summary.items():
    delta_e = energy - energy_lib
    delta_f = np.abs(forces - forces_lib).max()
    print(
        f"{name:18s} energy = {energy: .12f} eV  "
        f"max |dF| = {delta_f:.3e} eV/Ang  dE = {delta_e:.3e} eV"
    )

np.testing.assert_allclose(energy_ann, energy_lib, atol=1e-8)
np.testing.assert_allclose(forces_ann, forces_lib, atol=1e-6)
np.testing.assert_allclose(energy_torch, energy_lib, atol=1e-8)
np.testing.assert_allclose(forces_torch, forces_lib, atol=1e-8)
np.testing.assert_allclose(energy_ase, energy_lib, atol=1e-8)
np.testing.assert_allclose(forces_ase, forces_lib, atol=1e-8)

LibAenetInterface  energy = -19517.694574626403 eV  max |dF| = 0.000e+00 eV/Ang  dE = 0.000e+00 eV
ANNPotential       energy = -19517.694574630001 eV  max |dF| = 4.867e-07 eV/Ang  dE = -3.598e-09 eV
TorchANNPotential  energy = -19517.694574626406 eV  max |dF| = 1.186e-11 eV/Ang  dE = -3.638e-12 eV
AenetCalculator    energy = -19517.694574626403 eV  max |dF| = 2.442e-15 eV/Ang  dE = 0.000e+00 eV


## 3. Geometry optimization with ASE

For structure relaxation, the ASE calculator interface is the most convenient option because it plugs directly into ASE's optimizers. The capped optimization below keeps this example lightweight while still reaching a modest force threshold.


In [4]:
from ase.optimize import BFGS

atoms_opt = cell_atoms.copy()
atoms_opt.calc = calculator

optimizer = BFGS(atoms_opt, logfile=None)
optimizer.run(fmax=0.2, steps=30)

opt_path = outdir + "/TiO2-cell-opt.xsf"
ase.io.write(opt_path, atoms_opt)

print(f"Optimized structure written to: {opt_path}")
print(f"Optimized energy: {atoms_opt.get_potential_energy():.12f} eV")
print(f"Final max |force|: {np.abs(atoms_opt.get_forces()).max():.6f} eV/Ang")

Optimized structure written to: ./example-08-outputs/TiO2-cell-opt.xsf
Optimized energy: -19521.558056537517 eV
Final max |force|: 0.184319 eV/Ang


## 4. Short molecular-dynamics example

ASE can also use the same calculator for molecular dynamics. The simulation below is intentionally short and is meant only as a minimal illustrative example on the optimized TiO$_2$ cell.


In [5]:
from ase import units
from ase.io.trajectory import Trajectory
from ase.md.velocitydistribution import (
    MaxwellBoltzmannDistribution,
    Stationary,
    ZeroRotation,
)
from ase.md.verlet import VelocityVerlet

atoms_md = ase.io.read(opt_path)
atoms_md.calc = calculator

MaxwellBoltzmannDistribution(atoms_md, temperature_K=300)
Stationary(atoms_md)
ZeroRotation(atoms_md)

timestep_fs = 1.0
steps_per_block = 5
num_blocks = 4

trajectory_path = outdir + "/TiO2-cell-md.traj"
dynamics = VelocityVerlet(atoms_md, timestep_fs * units.fs)
trajectory = Trajectory(trajectory_path, "w", atoms_md)
dynamics.attach(trajectory.write, interval=steps_per_block)

print("Running a short NVE trajectory on the optimized TiO2 cell")
for block in range(num_blocks):
    dynamics.run(steps_per_block)
    print(
        f"block {block + 1:02d}: "
        f"E_pot = {atoms_md.get_potential_energy():.6f} eV, "
        f"E_kin = {atoms_md.get_kinetic_energy():.6f} eV, "
        f"T = {atoms_md.get_temperature():.2f} K"
    )

trajectory.close()
print(f"Trajectory written to: {trajectory_path}")


Running a short NVE trajectory on the optimized TiO2 cell
block 01: E_pot = -19521.435960 eV, E_kin = 0.823830 eV, T = 277.11 K
block 02: E_pot = -19521.172262 eV, E_kin = 0.560617 eV, T = 188.57 K
block 03: E_pot = -19520.948340 eV, E_kin = 0.337040 eV, T = 113.37 K
block 04: E_pot = -19520.891456 eV, E_kin = 0.280080 eV, T = 94.21 K
Trajectory written to: ./example-08-outputs/TiO2-cell-md.traj


## Takeaways

- `LibAenetInterface` gives direct Python access to `libaenet` for energy and force evaluation.
- `AenetCalculator` is the most convenient route when you want ASE optimizers or dynamics.
- The TiO$_2$ example shows that all available interfaces are numerically consistent for the same potential.
